In [1]:
#!/usr/bin/env python3
"""
QuantX Grid Bot — Standalone Test Script
Run this in VS Code or terminal to test the full grid bot pipeline
WITHOUT deploying to Railway.

Requirements:
    pip install longport-openapi

Usage:
    python test_grid_bot.py

What it tests:
    1. LongPort connection (QuoteContext + TradeContext)
    2. Quote fetch for 700.HK
    3. Grid initialization (places real limit orders)
    4. Order fill callback wiring
    5. Clean shutdown

Set your credentials below before running.
"""

import os, sys, math, json, time, decimal, logging, threading
from datetime import datetime, timezone, timedelta

# ── YOUR CREDENTIALS — fill these in ─────────────────────────────────────────
APP_KEY      = "6e73506603d52be23b36cc884ca6fd8b"
APP_SECRET   = "66785f824db9072cf288d52ea0a1ca6649f3fddb61d4f0a01534277ce2500c58"
ACCESS_TOKEN = "m_eyJhbGciOiJSUzI1NiIsImtpZCI6ImQ5YWRiMGIxYTdlNzYxNzEiLCJ0eXAiOiJKV1QifQ.eyJpc3MiOiJsb25nYnJpZGdlIiwic3ViIjoiYWNjZXNzX3Rva2VuIiwiZXhwIjoxNzg0Njc5MzE3LCJpYXQiOjE3NzY5MDMzMTYsImFrIjoiNmU3MzUwNjYwM2Q1MmJlMjNiMzZjYzg4NGNhNmZkOGIiLCJhYWlkIjoyMTMxNDk3NSwiYWMiOiJsYl9zZyIsIm1pZCI6MjY4NjE0MDcsInNpZCI6IjMwK0xWMlpFSHhaY01qeHl3NEpIeFE9PSIsImJsIjozLCJ1bCI6MCwiaWsiOiJsYl9zZ18yMTMxNDk3NSJ9.5e4x5MHtSwXQ7Uw4e_kUUDjA-SPrbsP8YWzmtspIWRtpn7KaF00Z-fPOUTeLITfoWLl1i1JnUdix8Ka2IPbaC8t6Ns6VPvwKLpFs0Opv86Bx-RdVZiGwZ92prs0GNh47LVUTpOpoWvPYa0RvNTuyQNTkrx2j-fMXG1jd037dlUqgRp-mxjA5ePqTtC-VUhmkOWg3mGCfrGmRJR45ZVo-Q7mZuDCVm_9eGej5ZdwobOL49-rgStFXs3FKaYNY67Rtu-ZKL2Segi9-P3DMac80OrwEEue6Nf_2_Bg0i4HCuEQBG_ynMIzqMydcO5FLLcv-9TlG6-Hapfdwnd49aPGhkL2opDNlZyRKemXcKtyfCk1hf_0VL5BM1T4fAvNVlosDYQEp2Uk2UShuMiUWueKkNy49gjbbClWTx5qalMGJXZ37U6aJgMc75kBH9K2PHH1DWyEKb4VNv_hYwpx_RgmABytNGHEGDRH5Slx7eW6I1o6yQcjq0yl_uw9oMptFWDt5wYVdU2mKH4tdm-q0OYCO1OiSI9ji-vkXSBGy3PEavvsqq9u3yfl87IfcESmcrp5V7Rfco2CXmwHoyZziI1ad_oRC3wygU4Gx8utBgmbWhEOWkW6Po9j6Z5GEPqWHqpxwp6S69Xxl4Z1I_en0GuWffVIQojPFTnH9SbHEKqY3eio"

# ── TEST PARAMETERS ───────────────────────────────────────────────────────────
SYMBOL          = "700.HK"
GRID_LEVELS     = 2          # small for testing
GRID_SPACING_PCT = 0.5       # 0.5% between levels
TP_PCT          = 0.3        # 0.3% take profit per level
SL_BOUNDARY_PCT = 5.0        # 5% stop loss boundary
LOT_SIZE        = 100        # HK minimum board lot
DRY_RUN         = True       # ← SET TO False FOR LIVE ORDERS

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger('grid-test')

SGT = timezone(timedelta(hours=8))
_shutdown = threading.Event()

# ── HK Tick rounding ──────────────────────────────────────────────────────────
def _hk_tick(p):
    if p < 0.25:   return 0.001
    elif p < 0.5:  return 0.005
    elif p < 10:   return 0.010
    elif p < 20:   return 0.020
    elif p < 100:  return 0.050
    elif p < 200:  return 0.100
    elif p < 500:  return 0.200
    elif p < 1000: return 0.500
    elif p < 2000: return 1.000
    elif p < 5000: return 2.000
    else:          return 5.000

def hk_tick_round(p):
    tick = _hk_tick(p)
    return round(math.floor(p / tick) * tick, 4)

def hk_tick_round_sell(p):
    tick = _hk_tick(p)
    return round(math.ceil(p / tick) * tick, 4)


# ── STEP 1: Connect to LongPort ───────────────────────────────────────────────
def test_connection():
    logger.info("=" * 60)
    logger.info("STEP 1: Testing LongPort connection")
    logger.info("=" * 60)
    try:
        from longport.openapi import Config, QuoteContext, TradeContext
        cfg = Config(app_key=APP_KEY, app_secret=APP_SECRET, access_token=ACCESS_TOKEN)
        quote_ctx = QuoteContext(cfg)
        logger.info("✓ QuoteContext connected")
        if not DRY_RUN:
            trade_ctx = TradeContext(cfg)
            logger.info("✓ TradeContext connected")
        else:
            trade_ctx = None
            logger.info("✓ DRY RUN: TradeContext skipped")
        return cfg, quote_ctx, trade_ctx
    except Exception as e:
        logger.error(f"✗ Connection failed: {e}")
        raise


# ── STEP 2: Fetch live quote ──────────────────────────────────────────────────
def test_quote(quote_ctx):
    logger.info("=" * 60)
    logger.info("STEP 2: Fetching live quote for %s", SYMBOL)
    logger.info("=" * 60)
    try:
        quotes = quote_ctx.quote([SYMBOL])
        price = float(quotes[0].last_done)
        logger.info("✓ %s current price: HK$%.4f", SYMBOL, price)
        logger.info("  Open: %.4f  High: %.4f  Low: %.4f  Volume: %s",
                    float(quotes[0].open), float(quotes[0].high),
                    float(quotes[0].low), quotes[0].volume)
        return price
    except Exception as e:
        logger.error(f"✗ Quote fetch failed: {e}")
        raise


# ── STEP 3: Build grid levels ─────────────────────────────────────────────────
def build_grid(cmp):
    logger.info("=" * 60)
    logger.info("STEP 3: Building grid around CMP=%.4f", cmp)
    logger.info("=" * 60)
    spacing = cmp * GRID_SPACING_PCT / 100.0
    levels = []
    for i in range(1, GRID_LEVELS + 1):
        entry = hk_tick_round(cmp - i * spacing)
        tp    = hk_tick_round_sell(entry * (1 + TP_PCT / 100))
        sl    = hk_tick_round(cmp * (1 - SL_BOUNDARY_PCT / 100))
        levels.append({'level': i, 'entry': entry, 'tp': tp, 'sl_boundary': sl})
        logger.info("  [BUY_%d] entry=HK$%.4f  TP=HK$%.4f  SL_boundary=HK$%.4f",
                    i, entry, tp, sl)
    logger.info("  Spacing: %.2f%% = HK$%.4f per level", GRID_SPACING_PCT, spacing)
    logger.info("  Capital required: HK$%.2f (approx)", levels[0]['entry'] * LOT_SIZE * GRID_LEVELS)
    return levels


# ── STEP 4: Place orders (or simulate) ───────────────────────────────────────
def place_orders(quote_ctx, trade_ctx, levels):
    logger.info("=" * 60)
    logger.info("STEP 4: Placing %d BUY limit orders (%s)",
                len(levels), "DRY RUN" if DRY_RUN else "LIVE")
    logger.info("=" * 60)

    if DRY_RUN:
        for lv in levels:
            logger.info("  [DRY RUN] Would place LIMIT BUY %d %s @ HK$%.4f",
                        LOT_SIZE, SYMBOL, lv['entry'])
        logger.info("✓ DRY RUN complete — no real orders placed")
        return []

    # LIVE orders
    from longport.openapi import OrderType, OrderSide, TimeInForceType
    order_ids = []
    for lv in levels:
        try:
            resp = trade_ctx.submit_order(
                symbol=SYMBOL,
                order_type=OrderType.LO,
                side=OrderSide.Buy,
                submitted_quantity=LOT_SIZE,
                time_in_force=TimeInForceType.Day,
                submitted_price=decimal.Decimal(str(lv['entry'])),
                remark=f"QuantX Grid Test BUY_{lv['level']}"
            )
            oid = str(resp.order_id)
            order_ids.append(oid)
            logger.info("  ✓ BUY_%d placed: oid=%s @ HK$%.4f",
                        lv['level'], oid, lv['entry'])
            time.sleep(0.3)  # avoid rate limit
        except Exception as e:
            logger.error("  ✗ BUY_%d failed: %s", lv['level'], e)
    return order_ids


# ── STEP 5: Check open orders ─────────────────────────────────────────────────
def check_orders(trade_ctx, order_ids):
    if DRY_RUN or not order_ids:
        return
    logger.info("=" * 60)
    logger.info("STEP 5: Checking order status")
    logger.info("=" * 60)
    try:
        from longport.openapi import OrderStatus
        for oid in order_ids:
            order = trade_ctx.order_detail(order_id=oid)
            logger.info("  Order %s: status=%s price=%.4f qty=%d",
                        oid, order.status, float(order.price or 0), int(order.quantity or 0))
    except Exception as e:
        logger.error("Order check failed: %s", e)


# ── STEP 6: Wire fill callback (LIVE only) ────────────────────────────────────
def wire_fill_callback(trade_ctx, levels):
    if DRY_RUN or not trade_ctx:
        logger.info("STEP 6: Skipping fill callback (DRY RUN)")
        return

    logger.info("=" * 60)
    logger.info("STEP 6: Wiring order fill callback")
    logger.info("=" * 60)

    from longport.openapi import OrderStatus, OrderSide, TopicType

    order_to_level = {}

    def on_order_changed(event):
        if event.status not in (OrderStatus.Filled, OrderStatus.PartialFilled):
            return
        oid = str(event.order_id)
        fill_price = float(event.executed_price or 0)
        fill_qty = int(event.executed_quantity or 0)
        side = "BUY" if event.side == OrderSide.Buy else "SELL"
        logger.info("🔔 FILL: %s %d %s @ HK$%.4f | oid=%s",
                    side, fill_qty, SYMBOL, fill_price, oid)

        if oid in order_to_level:
            lv = order_to_level[oid]
            tp_price = hk_tick_round_sell(fill_price * (1 + TP_PCT / 100))
            logger.info("  → Placing TP SELL %d @ HK$%.4f", fill_qty, tp_price)
            try:
                from longport.openapi import OrderType, TimeInForceType
                resp = trade_ctx.submit_order(
                    symbol=SYMBOL, order_type=OrderType.LO,
                    side=OrderSide.Sell, submitted_quantity=fill_qty,
                    time_in_force=TimeInForceType.Day,
                    submitted_price=decimal.Decimal(str(tp_price)),
                    remark=f"QuantX Grid Test TP_{lv['level']}"
                )
                logger.info("  ✓ TP SELL placed: oid=%s", resp.order_id)
            except Exception as e:
                logger.error("  ✗ TP SELL failed: %s", e)

    trade_ctx.set_on_order_changed(on_order_changed)
    trade_ctx.subscribe([TopicType.Private])
    logger.info("✓ Fill callback wired")


# ── STEP 7: Cancel all orders (cleanup) ───────────────────────────────────────
def cancel_all(trade_ctx, order_ids):
    if DRY_RUN or not order_ids:
        return
    logger.info("=" * 60)
    logger.info("CLEANUP: Cancelling %d open orders", len(order_ids))
    logger.info("=" * 60)
    for oid in order_ids:
        try:
            trade_ctx.cancel_order(oid)
            logger.info("  ✓ Cancelled: %s", oid)
        except Exception as e:
            logger.warning("  Could not cancel %s: %s", oid, e)


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    logger.info("=" * 60)
    logger.info("QuantX Grid Bot — Standalone Test")
    logger.info("Symbol: %s | DRY_RUN: %s", SYMBOL, DRY_RUN)
    logger.info("Grid levels: %d | Spacing: %.1f%% | TP: %.1f%% | SL: %.1f%%",
                GRID_LEVELS, GRID_SPACING_PCT, TP_PCT, SL_BOUNDARY_PCT)
    logger.info("=" * 60)

    if APP_KEY == "YOUR_APP_KEY_HERE":
        logger.error("❌ Please fill in your LongPort credentials at the top of this file!")
        return

    order_ids = []
    trade_ctx = None

    try:
        cfg, quote_ctx, trade_ctx = test_connection()
        cmp = test_quote(quote_ctx)
        levels = build_grid(cmp)
        wire_fill_callback(trade_ctx, levels)
        order_ids = place_orders(quote_ctx, trade_ctx, levels)
        check_orders(trade_ctx, order_ids)

        if DRY_RUN:
            logger.info("=" * 60)
            logger.info("✅ ALL STEPS PASSED (DRY RUN)")
            logger.info("   Connection: OK")
            logger.info("   Quote fetch: OK  (%.4f)", cmp)
            logger.info("   Grid calc: OK  (%d levels)", len(levels))
            logger.info("   Order simulation: OK")
            logger.info("")
            logger.info("   → Set DRY_RUN = False to place real orders")
            logger.info("=" * 60)
        else:
            logger.info("=" * 60)
            logger.info("✅ LIVE ORDERS PLACED")
            logger.info("   %d limit buy orders active on LongPort", len(order_ids))
            logger.info("   Check your LongPort app to confirm")
            logger.info("")
            logger.info("   Press Ctrl+C to cancel all orders and exit")
            logger.info("=" * 60)
            # Wait for fills
            try:
                while not _shutdown.is_set():
                    _shutdown.wait(5)
                    # Re-check orders periodically
                    check_orders(trade_ctx, order_ids)
            except KeyboardInterrupt:
                pass

    except Exception as e:
        logger.error("Test failed: %s", e)
        import traceback
        traceback.print_exc()
    finally:
        if order_ids and not DRY_RUN:
            ans = input("\nCancel all open orders? (y/n): ")
            if ans.lower() == 'y':
                cancel_all(trade_ctx, order_ids)
        logger.info("Test complete.")


if __name__ == "__main__":
    main()

2026-05-05 10:42:03,087 | INFO | ============================================================
2026-05-05 10:42:03,089 | INFO | QuantX Grid Bot — Standalone Test
2026-05-05 10:42:03,089 | INFO | Symbol: 700.HK | DRY_RUN: True
2026-05-05 10:42:03,092 | INFO | Grid levels: 2 | Spacing: 0.5% | TP: 0.3% | SL: 5.0%
2026-05-05 10:42:03,092 | INFO | ============================================================
2026-05-05 10:42:03,093 | INFO | ============================================================
2026-05-05 10:42:03,094 | INFO | STEP 1: Testing LongPort connection
2026-05-05 10:42:03,095 | INFO | ============================================================
2026-05-05 10:42:05,791 | INFO | ✓ QuoteContext connected
2026-05-05 10:42:05,793 | INFO | ✓ DRY RUN: TradeContext skipped
2026-05-05 10:42:05,794 | INFO | ============================================================
2026-05-05 10:42:05,794 | INFO | STEP 2: Fetching live quote for 700.HK
2026-05-05 10:42:05,795 | INFO | ================